# Bước 1: Cài đặt thư viện
Chạy ô này để cài đặt thư viện giao tiếp với Azure và kiểm tra xem máy có nhận GPU không.

In [ ]:
!pip install azure-storage-blob
!nvidia-smi

# Bước 2: Khai báo cấu hình
Điền thông tin kết nối Azure của bạn vào đây.

In [ ]:
AZURE_CONNECTION_STRING = "DefaultEndpointsProtocol=https;AccountName=aicvideos;AccountKey=ZLdH5hcKb213WiCzOLCmfLrPRWR3rVkDdlfwV5iXr2snbY9lAMAJc9ZlHm9salxNTAvhcaNVJh0N+ASt+nNulQ==;EndpointSuffix=core.windows.net"
CONTAINER_NAME = "videos"

# Đổi đường dẫn này thành thư mục chứa video gốc trên Kaggle của bạn (Batch 2)
INPUT_VIDEO_DIR = "/kaggle/input/aic25-btc-b2-v1/videos" 

# Nơi lưu file tạm trước khi up lên Azure (sẽ bị xóa ngay sau khi up để trống ổ)
TEMP_DIR = "/kaggle/working/proxy_videos"

# Bước 3: Hàm xử lý FFmpeg (720p + Fast Start)
Hàm này dùng phần cứng GPU `h264_nvenc` để nén video siêu tốc.

In [ ]:
import os
import subprocess
from pathlib import Path

Path(TEMP_DIR).mkdir(parents=True, exist_ok=True)

def process_video_ffmpeg(input_path, output_path):
    """
    Ép xuống 720p, bitrate ~1M, gắn cờ faststart, nén bằng GPU h264_nvenc.
    Nếu máy không có GPU thì đổi 'h264_nvenc' thành 'libx264'.
    """
    command = [
        "ffmpeg", "-y",             # -y: Ghi đè file nếu có
        "-hwaccel", "cuda",         # Tăng tốc giải mã bằng CUDA
        "-i", str(input_path),
        "-vf", "scale=-2:720",      # Chiều cao 720p, chiều rộng tự co giãn
        "-c:v", "h264_nvenc",       # Encode bằng lõi GPU của Nvidia
        "-preset", "p4",            # Preset tốc độ cân bằng
        "-b:v", "1M",               # Ép dung lượng hình ảnh xuống thấp
        "-c:a", "aac", "-b:a", "128k",# Nén âm thanh
        "-movflags", "faststart",   # TỐI QUAN TRỌNG: Dời mục lục lên đầu để tua nhanh
        str(output_path)
    ]
    
    try:
        subprocess.run(command, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
        return True
    except subprocess.CalledProcessError as e:
        print(f"Lỗi FFmpeg: {e.stderr.decode('utf-8', errors='ignore')}")
        return False

# Bước 4: Khởi chạy Upload hàng loạt
Vòng lặp quét video -> Nén bằng GPU -> Up Azure -> Xóa file tạm.

In [ ]:
from azure.storage.blob import BlobServiceClient
import time
import glob

# Khởi tạo kết nối Azure
blob_service_client = BlobServiceClient.from_connection_string(AZURE_CONNECTION_STRING)
container_client = blob_service_client.get_container_client(CONTAINER_NAME)

# Lọc ra tất cả file mp4
video_files = glob.glob(os.path.join(INPUT_VIDEO_DIR, "*.mp4"))
video_files.sort()
print(f"Tìm thấy {len(video_files)} video cần xử lý.")

for i, input_file in enumerate(video_files, 1):
    video_name = os.path.basename(input_file)
    temp_file = os.path.join(TEMP_DIR, video_name)
    
    print(f"[{i}/{len(video_files)}] Đang nén GPU cho: {video_name} ...")
    start_time = time.time()
    
    # 1. Nén và tối ưu Video
    success = process_video_ffmpeg(input_file, temp_file)
    if not success:
        print(f"  -> Bỏ qua {video_name} do lỗi nén.")
        continue
        
    # 2. Upload lên Azure
    print(f"  -> Đang bơm lên Azure...")
    blob_client = container_client.get_blob_client(video_name)
    with open(temp_file, "rb") as data:
        # overwrite=True để lỡ đứt gánh up lại không bị lỗi
        blob_client.upload_blob(data, overwrite=True)
        
    # 3. Xóa rác, giải phóng ổ cứng Kaggle
    os.remove(temp_file)
    
    elapsed = time.time() - start_time
    print(f"  -> XONG {video_name} (mất {elapsed:.1f} giây).")

print("HOÀN TẤT TOÀN BỘ!")